In [1]:
import numpy as np
import pandas as pd

In [2]:
def backtest(data, stopLoss, takeProfit):
    total_value = 10000
    returns = []
    holdingDays = []

    for row in data.itertuples():
        acquisition_price = getattr(row, 'p1', None)
        if acquisition_price is None or pd.isna(acquisition_price):
            continue

        profit_loss = None
        daysHeld = 19

        for i in range(2,21):
            price = getattr(row, f"p{i}", None)
            if price is None or pd.isna(price):
                continue
            if price <= acquisition_price*(1-stopLoss):
                profit_loss = -stopLoss
                daysHeld = i
                break
            elif price >= acquisition_price*(1+takeProfit):
                profit_loss = takeProfit
                daysHeld = i
                break
        if profit_loss is None:
            price_Day20 = getattr(row, "p20", None)
            if price_Day20 is not None and not pd.isna(price_Day20):
                profit_loss = (price_Day20 - acquisition_price) / acquisition_price
            else:
                profit_loss = 0 
             
        returns.append(profit_loss)
        holdingDays.append(daysHeld)
        total_value =   total_value + 10000 * profit_loss

    if (len(returns)==0):
        return {
        "Total Return": 0,
        "Sharpe Ratio": -np.inf,
        "Win Rate": 0,
        "Average Holding Days": 0,
        "Profit Factor": 0,
        "Total Trades": 0,
        }

    returns = np.array(returns)
    holdingDays = np.array(holdingDays)

    avg_returns = np.mean(returns)
    std_returns = np.std(returns)
    sharpe = (avg_returns/std_returns) if std_returns > 0 else -np.inf
    # totalPercReturn = (total_value - 10000)/ 10000
    totalPercReturn = np.sum(returns)

    wins = returns > 0
    losses = returns < 0
    winRate = np.mean(wins)
    avgHoldingDays = np.mean(holdingDays)

    totalGains = np.sum(returns[wins])
    totalLosses = - np.sum(returns[losses])
    profitFactor = (totalGains / totalLosses) if totalLosses > 0 else np.inf


    results = {
        "Total Return": totalPercReturn,
        "Sharpe Ratio": sharpe,
        "Win Rate": winRate,
        "Average Holding Days": avgHoldingDays,
        "Profit Factor": profitFactor,
        "Total Trades": len(returns),
    }

    return results

In [3]:
def optimal_triggers(data):
    data = data[data['NCAR20_quintiles'].astype(float) == 1]
    print(data.shape)
    stopLoss_values = np.round(np.arange(0.05, 0.35, 0.01),2)
    takeProfit_values = np.round(np.arange(0.05, 0.50, 0.01),2)

    results = []

    for sL in stopLoss_values:
        for tP in takeProfit_values:
            r = backtest(data, sL, tP)
            results.append({'StopLoss': sL, 
                            'TakeProfit': tP, 
                            'TotalReturn': r['Total Return'], 
                            'SharpeRatio': r['Sharpe Ratio'], 
                            'WinRate': r['Win Rate'], 
                            'AvgDays': r['Average Holding Days'],
                            'ProfitFactor': r['Profit Factor'],
                            'TradeCount': r['Total Trades']
                            })
            
    results_df = pd.DataFrame(results)
    optimal_idx = results_df['SharpeRatio'].idxmax()
    optimal_row = results_df.loc[optimal_idx]

    optimal_results = {
        'OptimalProfit': optimal_row['TotalReturn'], 
        'Optimal StopLoss': optimal_row['StopLoss'], 
        'Optimal TakeProfit': optimal_row['TakeProfit'], 
        'Optimal Sharpe': optimal_row['SharpeRatio'], 
        'Optimal WinRate': optimal_row['WinRate'], 
        'Optimal AvgDays': optimal_row['AvgDays']
    }

    return optimal_results, results_df

In [4]:
final_df = pd.read_csv("datasets/final_ds.csv")
final_df_train = final_df[(final_df['date'] >= '2025-01-01') & (final_df['date'] <= '2025-11-30')]

prms, grid = optimal_triggers(final_df_train)
optimal_SL = prms['Optimal StopLoss']
optimal_TP = prms['Optimal TakeProfit']
optimal_Sharpe = prms['Optimal Sharpe']
print(optimal_SL, optimal_TP, optimal_Sharpe)

(425, 63)
0.13 0.25 0.7132013596204619


In [6]:
final_df_test = final_df[(final_df['date'] >= '2026-01-01') & (final_df['date'] <= '2026-06-30')]
final_df_test = final_df_test[final_df_test['NCAR20_quintiles'].astype(float) == 1]
print(backtest(final_df_test, optimal_SL, optimal_TP))

{'Total Return': np.float64(29.17573381665194), 'Sharpe Ratio': np.float64(0.6753659718190909), 'Win Rate': np.float64(0.7054794520547946), 'Average Holding Days': np.float64(14.11986301369863), 'Profit Factor': np.float64(4.570121185180584), 'Total Trades': 292}
